# Shreve Week 13 — Exotic Options

**Week 13** — Exotic Options

This notebook teaches **exotic options** in the style of our graduate probability notebook: definitions from **Shreve**, intuition and examples from **Baxter & Rennie**, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve | Baxter & Rennie |
|------|-------|--------|-----------------|
| 1 | **Barrier options** | Ch. 7.1 | Ch. 4.1 |
| 2 | **Asian options** | Ch. 7.2 | Ch. 4.2 |
| 3 | **Lookback options** | Ch. 7.3 | Ch. 4.3 |
| 4 | **Path-dependent payoffs** | Ch. 7 | Ch. 4.5 |
| — | **Problem set** | Ch. 7 | App. 3 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. **Shreve** (*Stochastic Calculus for Finance II*) — rigorous measure-theoretic treatment; see chapter pointers in each section.
5. **Baxter & Rennie** (*Financial Calculus*) — market intuition, replication, and worked examples; see spotlight sections.

**References:**
- **Shreve** Vol II — Ch. 7 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Baxter & Rennie**, *Financial Calculus* — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Barrier Options

**Down-and-out call:** pays $(S_T-K)^+$ only if $S_t > B$ for all $t \leq T$.

Cheaper than vanilla — knockout reduces value.

**Shreve Ch. 7.1:** barrier options via reflection / PDE.


In [ ]:
def simulate_barrier_call(S0, K, B, r, sigma, T, n_steps, seed=130):
    rng = np.random.default_rng(seed)
    dt = T/n_steps
    S = S0
    alive = True
    for _ in range(n_steps):
        dW = rng.normal(0, np.sqrt(dt))
        S = S * np.exp((r-0.5*sigma**2)*dt + sigma*dW)
        if S <= B:
            alive = False
            break
    payoff = max(S-K, 0) if alive else 0
    return payoff

S0, K, B = 100, 100, 80
payoffs = [simulate_barrier_call(S0, K, B, 0.05, 0.2, 1, 252, seed=s)
           for s in range(50_000)]
barrier_price = np.exp(-0.05)*np.mean(payoffs)
print(f"Down-and-out call (B=80) ≈ {barrier_price:.4f}")


---
# Part 2 — Asian Options

Payoff depends on **average** price: $(\bar{S} - K)^+$ where $\bar{S} = \frac{1}{n}\sum S_{t_i}$.

Path-dependent; no closed form for arithmetic average under GBM.

**Shreve Ch. 7.2:** Asian options — MC pricing.


In [ ]:
def asian_call_mc(S0, K, r, sigma, T, n_steps, n_sims=50_000, seed=131):
    rng = np.random.default_rng(seed)
    dt = T/n_steps
    prices = []
    for _ in range(n_sims):
        S = S0
        path = []
        for _ in range(n_steps):
            dW = rng.normal(0, np.sqrt(dt))
            S = S*np.exp((r-0.5*sigma**2)*dt + sigma*dW)
            path.append(S)
        avg_S = np.mean(path)
        prices.append(max(avg_S-K, 0))
    return np.exp(-r*T)*np.mean(prices)

asian = asian_call_mc(100, 100, 0.05, 0.2, 1, 252)
print(f"Asian call price ≈ {asian:.4f}")


---
# Part 3 — Lookback Options

**Floating strike lookback call:** payoff $S_T - S_{\min}$.

Benefit from lowest price over horizon.

**Shreve Ch. 7.3:** lookback options.


In [ ]:
def lookback_call_mc(S0, r, sigma, T, n_steps, n_sims=30_000, seed=132):
    rng = np.random.default_rng(seed)
    dt = T/n_steps
    payoffs = []
    for _ in range(n_sims):
        S = S0
        S_min = S0
        for _ in range(n_steps):
            dW = rng.normal(0, np.sqrt(dt))
            S = S*np.exp((r-0.5*sigma**2)*dt + sigma*dW)
            S_min = min(S_min, S)
        payoffs.append(S - S_min)
    return np.exp(-r*T)*np.mean(payoffs)

lookback = lookback_call_mc(100, 0.05, 0.2, 1, 252)
print(f"Lookback call ≈ {lookback:.4f}")


---
# Part 4 — Path Dependence & MC

Exotics require **full path simulation** — Feynman-Kac on augmented state or direct MC under $Q$.

**Shreve Ch. 7:** general path-dependent derivatives.


In [ ]:
# Vanilla vs Asian vs Barrier comparison
vanilla_mc = np.exp(-0.05)*np.mean([
    max(100*np.exp((0.05-0.5*0.2**2)*1 + 0.2*np.sqrt(1)*np.random.default_rng(s).standard_normal()), 100)-100
    for s in range(50_000)])
print(f"Vanilla call  ≈ {vanilla_mc:.4f}")
print(f"Asian call    ≈ {asian:.4f}")
print(f"Barrier call  ≈ {barrier_price:.4f}")
print(f"Lookback call ≈ {lookback:.4f}")


**Your turn:** Why is Asian call cheaper than vanilla for ATM options?


---
## Baxter & Rennie spotlight — Pricing market securities (Ch. 4)

B&R apply the same **change-of-measure** machinery to:

- **FX options** (Ch. 4.1): two currencies, two drifts, one risk-neutral measure.
- **Dividends** (Ch. 4.2): adjust spot for yield.
- **Bonds** (Ch. 4.3): forward bond prices as martingales.
- **Quantos** (Ch. 4.5): payoff in wrong currency — exotic structure.

Path-dependent exotics (Asian, barrier) are natural **Monte Carlo** extensions of Shreve Ch. 7; B&R Ch. 4 gives cross-asset templates.

**Read:** B&R Ch. 4.1–4.3 and 4.5.


**B&R Example (Ch. 4.1):** FX forward — under domestic risk-neutral measure, foreign bond discounted at foreign rate is a martingale when converted.


In [ ]:
# B&R Ch. 4.1 — FX: domestic vs foreign growth rates
r_d, r_f, T = 0.05, 0.03, 1.0
# Forward F_0 = S_0 * exp((r_d - r_f) * T)
S0 = 1.20  # USD per EUR
F = S0 * np.exp((r_d - r_f) * T)
print(f"FX forward F_0 = {F:.4f} (USD per EUR)")


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Define down-and-in put and its relationship to down-and-out put.

2. Price arithmetic Asian call by MC with 95% CI.

3. How does barrier level $B$ affect down-and-out call price?

4. Compare lookback vs vanilla call at $S_0=100$.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Down-and-in pays if barrier hit; in+out ≈ vanilla by parity on barrier segment.

**2.** MC with $n$ paths, CI $\hat{V} \pm 1.96\, \hat{\sigma}/\sqrt{n}$.

**3.** Lower $B$ (closer to $S_0$) → higher knockout prob → lower price.

**4.** Lookback > vanilla: guaranteed positive payoff $S_T - S_{\min} \geq 0$.

</details>


---
## Further reading

### Shreve
- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 7 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

### Baxter & Rennie
- **Baxter & Rennie**, Ch. 4 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf): FX, dividends, bonds, quantos.
- Shreve Ch. 7 exotics + B&R Ch. 4 cross-asset pricing = full practitioner toolkit.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
